# Causality Dataset Creation Functions

Using causality_dataset_creation_EDA.ipynb to create the final functions


Model 8:
1. Synapsida
2. Reptilia All

Model 9:
1. Synapsida
2. Reptilia Terres

Dataset format:
1. Genera name: one row per species
2. Lat range value for that species
3. Biome presence binary variables
4. ts: from combined combined_10_se_est.txts
5. te: from combined combined_10_se_est.txts
6. Age range: ts-te
7. Speciation rate: lam values from combined_10_per_species_rates.log. Remove 10% first iterations, calculate average per species
8. Extinction rate: mu values from combined_10_per_species_rates.log. Remove 10% first iterations, calculate average per species
9. Each climate variable (model 8 = isotopic, model 9 = BRIDGE), but per species by finding the climate variable's average value for that species' lifespan
    9.a. Nearest Neighbors (_nn) Method
    9.b. Weighted Interpolation (_wi) Method
10. Time Bin Column: Assigning each occurrence the time bin the exist in, with a 1myr margin. Each genera that spans multiple bins will be marked as such

## 1-8: Genera Data Function

Perform sections 1-8

In [1]:
import pandas as pd
import numpy as np

In [50]:
# Need three input datasets: lats_file_final.csv, combined_10_se_est.txt and combined_10_per_species_rates.log

def genera_data(path_to_lats_file, path_to_combined_se_est, path_to_per_species_rates):
    ####################################################################################################
    #################### Load datasets
    lats_file = pd.read_csv(path_to_lats_file)
    se_est = pd.read_csv(path_to_combined_se_est, sep="\t")
    per_species_rates = pd.read_csv(path_to_per_species_rates, sep="\t")

    ####################################################################################################
    #################### SECTIONS 1-6
    # Add a check that the "genus" column in the lats file and "species" column in se_est are identical in CONTENT (they have the exact number of the exact same species)
    assert set(lats_file['genus']) == set(se_est['species']), "Genus columns do not match in terms of CONTENT between se_est file and lats file."

    # Re-order the se_est and lats_file dataframes alphabetically, by the "species" and "genus" columns respectively
    se_est = se_est.sort_values(by='species').reset_index(drop=True)
    lats_file = lats_file.sort_values(by='genus').reset_index(drop=True)

    # Needs to pass a check that "genus" column in lats file and "species" columns in se_est are identical in order to continue
    assert se_est['species'].equals(lats_file['genus']), "Genus columns do not match between se_est file and lats file."

    # ts and te need + 175 added back to them b/c my BDNN runs had 175 subtracted from all time values
    se_est['ts'] = se_est['ts'] + 175
    se_est['te'] = se_est['te'] + 175
    
    # time_span column from se_est
    se_est['time_span'] = se_est['ts'] - se_est['te']

    # Merge datasets on 'genus' column
    merged_data = pd.merge(lats_file, se_est[['species', 'ts', 'te', 'time_span']], left_on='genus', right_on='species', how='left')
    merged_data.drop(columns=['species'], inplace=True)

    # Need to pass check that no rows were lost
    assert len(merged_data) == len(lats_file), "Row count mismatch after merging se_est file and lats file"

    ####################################################################################################
    #################### SECTIONS 7-8
    per_species_rates = per_species_rates.drop(columns=per_species_rates.columns[-1])  # Drop last unnamed column, all per species rates files have this
    
    # Drop 10% burn-in (first 100 rows, inclusive)
    per_species_rates_burned = per_species_rates.iloc[100:, :]

    # assert that the # of rows = 900 (all combined logs were resampled to 100 so they should all have 1000 rows originally, - 10% burn-in)
    assert len(per_species_rates_burned) == 900, "Row count incorrect after dropping 10 percent burn-in from per species rates file"

    # New dataset which = the average value of eah column, maintaining the column heads
    per_species_rates_mean = per_species_rates_burned.mean().to_frame().T

    # Make sure that no columns were lost
    assert len(per_species_rates_mean.columns) == len(per_species_rates.columns), "Avg per species rates df column count mismatch with original per species rates df"

    # Separate _lam and _mu columns into their own dataframes
    per_species_rates_mean_lam = per_species_rates_mean.filter(like='_lam')
    per_species_rates_mean_mu = per_species_rates_mean.filter(like='_mu')

    # Make sure we didn't lose any columns. The + 1 is for the iterations column that we lost in the filtering 
    assert len(per_species_rates_mean_lam.columns) + len(per_species_rates_mean_mu.columns) + 1 == len(per_species_rates_mean.columns), "Column count mismatch between _lam and _mu cols dfs and original mean df"
    assert len(per_species_rates_mean_lam.columns) == len(per_species_rates_mean_mu.columns), "_lam and _mu dfs column count mismatch"

    # Transpose to have species as rows and averages as columns
    per_species_rates_mean_lam_T = per_species_rates_mean_lam.T.reset_index()
    per_species_rates_mean_lam_T.rename(columns={0: 'avg_lam', 'index': 'species'}, inplace=True)
    per_species_rates_mean_mu_T = per_species_rates_mean_mu.T.reset_index()
    per_species_rates_mean_mu_T.rename(columns={0: 'avg_mu', 'index': 'species'}, inplace=True)

    # Strip the suffixes from the species names to prepare for merging
    per_species_rates_mean_lam_T['species'] = per_species_rates_mean_lam_T['species'].str.replace('_lam', '')
    per_species_rates_mean_mu_T['species'] = per_species_rates_mean_mu_T['species'].str.replace('_mu', '')

    # Length check
    assert per_species_rates_mean_lam_T.shape[0] == per_species_rates_mean_mu_T.shape[0] == merged_data.shape[0], "_lam, _mu, merged dfs row count mismatch"

    # Merge with main df
    merged_data = pd.merge(merged_data, per_species_rates_mean_lam_T, left_on='genus', right_on='species', how='left')
    merged_data = pd.merge(merged_data, per_species_rates_mean_mu_T, left_on='genus', right_on='species', how='left')
    merged_data.drop(columns=['species_x', 'species_y'], inplace=True)

    return merged_data


## 9: Climate Data Functions (2 Methods)

Perform section 9 in TWO ways:
1. Nearest Neighbor
2. Weighted Interpolation

See the causality_dataset_creation_EDA.ipynb file to see discussion of the pros/cons of each method

In [8]:
def nearest_neighbor(path_to_climate_data, merged_df):
    ####################################################################################################
    #################### Load climate dataset
    climate_data = pd.read_csv(path_to_climate_data, sep="\t")

    ####################################################################################################
    #################### SECTION 9
    original_row_count = len(merged_df)
    merged_df_nn = merged_df.copy()

    # Warning trackers if extrapolation or distant assignments occur (if a genera l's lifespan is outside climate data range or >1 Myr from midpoint, repsectively)
    extrapolated_genera = []
    distant_assignment_genera = []  # >1 Myr from midpoint
    climate_time_min = climate_data['Time'].min()
    climate_time_max = climate_data['Time'].max()
    
    # Automatically detect climate columns (all columns except 'Time')
    climate_cols = [col for col in climate_data.columns if col != 'Time']
    
    # Initialize new columns with _NN suffix
    for col in climate_cols:
        merged_df_nn[f'{col}_NN'] = np.nan
    
    for index, row in merged_df_nn.iterrows():
        ts = row['ts']
        te = row['te']
        
        # Filter climate data for the time span of the genus
        climate_filtered = climate_data[(climate_data['Time'] <= ts) & (climate_data['Time'] >= te)]
        
        # Calculate mean values for all climate columns
        mean_values = climate_filtered[climate_cols].mean() # Is this matematically poor if there are missing rows in isotopic data? Some myrs have no data
        
        for col in climate_cols:
            merged_df_nn.at[index, f'{col}_NN'] = mean_values[col]
        
        # If any null, assign values from closest time point to midpoint of lifespan
        if any(pd.isna(merged_df_nn.at[index, f'{col}_NN']) for col in climate_cols):
            midpoint = (ts + te) / 2
            closest_idx = (climate_data['Time'] - midpoint).abs().idxmin()
            closest_time = climate_data.loc[closest_idx]

            # Track if this is extrapolation (genus lifespan entirely outside climate data range)
            if ts < climate_time_min or te > climate_time_max:
                extrapolated_genera.append(row['genus'])
            
            # Track if nearest neighbor is >1 Myr away (very distant assignment)
            distance = abs(closest_time['Time'] - midpoint)
            if distance > 1.0:
                distant_assignment_genera.append((row['genus'], distance))
            
            for col in climate_cols:
                # Only replace if this specific column is null
                if pd.isna(merged_df_nn.at[index, f'{col}_NN']):
                    merged_df_nn.at[index, f'{col}_NN'] = closest_time[col]    

    assert len(merged_df_nn) == original_row_count, "The original dataframe and the nearest neighbor merged dataframe have different row counts."
    
    # Print warnings about extrapolation or distant assignments
    if extrapolated_genera:
        print(f"Warning: {len(extrapolated_genera)} genera had lifespans entirely outside climate data range "
              f"({climate_time_min}-{climate_time_max} Ma) and were assigned values via extrapolation: "
              f"{extrapolated_genera[:5]}{'...' if len(extrapolated_genera) > 5 else ''}")
    
    if distant_assignment_genera:
        distant_assignment_genera.sort(key=lambda x: x[1], reverse=True)
        print(f"Warning: {len(distant_assignment_genera)} genera were assigned climate values from time points >1 Myr from their midpoint. Most extreme cases: "
              f"{distant_assignment_genera[:5]}{'...' if len(distant_assignment_genera) > 5 else ''}")
        
    return merged_df_nn

In [11]:
def weighted_interpolation(path_to_climate_data, merged_df):
    ####################################################################################################
    #################### Load climate dataset
    climate_data = pd.read_csv(path_to_climate_data, sep="\t")

    ####################################################################################################
    #################### SECTION 10
    original_row_count = len(merged_df)
    merged_df_wi = merged_df.copy()

    # Warning trackers if extrapolation or distant assignments occur (if a genera l's lifespan is outside climate data range or >1 Myr from midpoint, repsectively)
    extrapolated_genera = []
    climate_time_min = climate_data['Time'].min()
    climate_time_max = climate_data['Time'].max()
    
    # Automatically detect climate columns (all columns except 'Time')
    climate_cols = [col for col in climate_data.columns if col != 'Time']
    
    # Initialize new columns with _WI suffix
    for col in climate_cols:
        merged_df_wi[f'{col}_WI'] = np.nan
    
    # Get sorted list of available time points for efficient lookup
    available_times = sorted(climate_data['Time'].unique(), reverse=True)
    
    for index, row in merged_df_wi.iterrows():
        ts = row['ts']  # Time of speciation (older)
        te = row['te']  # Time of extinction (younger)
        midpoint = (ts + te) / 2
        
        # Filter climate data for the time span of the genus
        climate_filtered = climate_data[(climate_data['Time'] <= ts) & (climate_data['Time'] >= te)]
        
        if len(climate_filtered) > 0:
            # Direct overlap exists - use arithmetic mean
            mean_values = climate_filtered[climate_cols].mean()
            for col in climate_cols:
                merged_df_wi.at[index, f'{col}_WI'] = mean_values[col]
        
        else:
            # No direct overlap - use weighted interpolation
            
            # Find t_upper: smallest time point >= ts (closest point older than or at speciation)
            upper_candidates = [t for t in available_times if t >= ts]
            t_upper = min(upper_candidates) if upper_candidates else available_times[0]
            
            # Find t_lower: largest time point <= te (closest point younger than or at extinction)
            lower_candidates = [t for t in available_times if t <= te]
            t_lower = max(lower_candidates) if lower_candidates else available_times[-1]

            # Track if this required extrapolation beyond climate data bounds
            if not upper_candidates or not lower_candidates:
                extrapolated_genera.append(row['genus'])
            
            # Calculate weights based on inverse distance to midpoint
            if t_upper != t_lower:
                w_lower = (t_upper - midpoint) / (t_upper - t_lower)
                w_upper = (midpoint - t_lower) / (t_upper - t_lower)
            else:
                # Edge case: both brackets are the same point (extrapolation)
                w_lower = 1.0
                w_upper = 0.0
            
            # Get climate values at the bracketing points
            lower_row = climate_data[climate_data['Time'] == t_lower].iloc[0]
            upper_row = climate_data[climate_data['Time'] == t_upper].iloc[0]
            
            # Calculate weighted averages for all climate columns
            for col in climate_cols:
                weighted_val = lower_row[col] * w_lower + upper_row[col] * w_upper
                merged_df_wi.at[index, f'{col}_WI'] = weighted_val
    
    assert len(merged_df_wi) == original_row_count, "The original dataframe and the weighted interpolation merged dataframe have different row counts."
    
    # Print warning
    if extrapolated_genera:
        print(f"Warning: {len(extrapolated_genera)} genera had lifespans that required extrapolation beyond "
              f"climate data range ({climate_time_min}-{climate_time_max} Ma). These genera were assigned values from the nearest available boundary: "
              f"{extrapolated_genera[:5]}{'...' if len(extrapolated_genera) > 5 else ''}")
        
    return merged_df_wi

## 10. Time Bins Function

1. Remove all genera that only exist outside of 274.4 and 237
2. New column that assigns bins below, but that can have ~1myr grace 
    - 274.4-264.3: Roadian-Wordian
    - 264.3-259.5: Capitanian
    - 259.5-252: Lopingian
    - 252-246.7: Early Triassic
    - 246.7-242: Anisian
    - 242-237: Ladinian

# Runs:

def genera_data(path_to_lats_file, path_to_combined_se_est, path_to_per_species_rates)
    - return merged_df

def nearest_neighbor(path_to_climate_data, merged_df)
    - return merged_df_nn

def weighted_interpolation(path_to_climate_data, merged_df)
    - return merged_df_wi

**NOTE CLIMATE DATA SHOULD BE A .TXT




### Model 8

In [5]:
isotopic_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\Perm-Trias\\songEA_isotopic_data_pt_1myr\\isotopic_1myr_filtered_final.txt"

In [12]:
model_8_syn_lats_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\Perm-Trias\\bdnn_trait_files\\pt_synapsida_lats_file_final.csv"
model_8_syn_se_est_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_8\\synapsida\\combined_10_se_est.txt"
model_8_syn_per_species_rates_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_8\\synapsida\\combined_10_per_species_rates.log"


model_8_syn = genera_data(model_8_syn_lats_file, 
            model_8_syn_se_est_file,
            model_8_syn_per_species_rates_file)

model_8_syn_nn = nearest_neighbor(isotopic_file, model_8_syn)

model_8_syn_nn_wi = weighted_interpolation(isotopic_file, model_8_syn_nn)

model_8_syn_nn_wi.head()

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lam,avg_mu,mean_pt_1myr_z_trans_NN,Mod_R_deltaTMyr_pt_1myr_z_trans_NN,mean_pt_1myr_z_trans_WI,Mod_R_deltaTMyr_pt_1myr_z_trans_WI
0,Abajudon,-0.297425,0,0,1,0,266.194269,259.871176,6.323094,1.482582,1.728257,-0.917951,0.704704,-0.917951,0.704704
1,Abdalodon,-0.297425,0,0,1,0,257.029067,256.833773,0.195294,1.096509,1.086128,-0.016673,-0.859216,-0.016673,-0.859216
2,Acratophorus,-0.297425,0,0,1,0,233.355524,228.597823,4.757701,0.469096,0.629002,1.005908,0.025418,1.005908,0.025418
3,Adelobasileus,-0.297425,0,1,0,0,216.763856,216.113424,0.650432,1.054359,1.021459,-0.162980,-0.763084,-0.074135,-0.093139
4,Aelurognathus,1.840695,0,0,1,0,260.112300,252.034343,8.077957,0.950114,0.964767,-0.544651,-0.226106,-0.544651,-0.226106


In [14]:
model_8_syn_nn_wi.to_csv("C:/Users/SimoesLabAdmin/Documents/pt_diversity_rates/updated_occurrence_analyses/data/causality_data/model_8_syn_merged_nn_wi.csv", index=False)

In [51]:
model_8_rep_lats_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\Perm-Trias\\bdnn_trait_files\\pt_reptilia_all_lats_file_final.csv"
model_8_rep_se_est_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_8\\reptilia_all_offset_biohpc\\combined_10_se_est.txt"
model_8_rep_per_species_rates_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_8\\reptilia_all_offset_biohpc\\combined_10_per_species_rates.log"

model_8_rep = genera_data(model_8_rep_lats_file, 
            model_8_rep_se_est_file,
            model_8_rep_per_species_rates_file)

model_8_rep_nn = nearest_neighbor(isotopic_file, model_8_rep)

model_8_rep_nn_wi = weighted_interpolation(isotopic_file, model_8_rep_nn)

model_8_rep_nn_wi.head()

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lam,avg_mu,mean_pt_1myr_z_trans_NN,Mod_R_deltaTMyr_pt_1myr_z_trans_NN,mean_pt_1myr_z_trans_WI,Mod_R_deltaTMyr_pt_1myr_z_trans_WI
0,Abyssomedon,-0.336825,0,1,0,0,287.772844,287.627814,0.145031,0.769772,0.264861,-0.571677,-0.692433,-0.272913,-0.186398
1,Acadiella,-0.336825,0,1,0,0,232.254143,231.276583,0.977560,1.481202,0.920088,1.568450,1.175279,1.568450,1.175279
2,Acaenasuchus,-0.336825,0,1,0,0,226.720959,209.640777,17.080182,1.096897,0.987912,0.270584,0.052530,0.270584,0.052530
3,Acallosuchus,-0.336825,0,1,0,0,216.764203,215.899536,0.864668,0.693004,0.871778,-0.162980,-0.763084,-0.162980,-0.763084
4,Acerosodontosaurus,-0.336825,0,0,1,0,255.180817,255.078708,0.102109,1.219460,1.272939,-1.039392,-0.146453,-0.976977,-0.129001


In [52]:
model_8_rep_nn_wi.to_csv("C:/Users/SimoesLabAdmin/Documents/pt_diversity_rates/updated_occurrence_analyses/data/causality_data/model_8_rep_merged_nn_wi.csv", index=False)

#### Troubleshoot

In [ ]:
# This code just shows that the genus columns in the two input files might be out of order
# Had to go back and add additional asserts in the genera_data function to ensure they are identical in content and order
model_8_rep_lats_file_test = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\Perm-Trias\\bdnn_trait_files\\pt_reptilia_all_lats_file_final.csv")
model_8_rep_se_est_file_test = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_8\\reptilia_all_offset_biohpc\\combined_10_se_est.txt", sep="\t")

# Print exactly which genera each dataset has that the other does not
set_lats = set(model_8_rep_lats_file_test['genus'])
set_se_est = set(model_8_rep_se_est_file_test['species'])
only_in_lats = set_lats - set_se_est
only_in_se_est = set_se_est - set_lats

print(f"Genera only in lats file ({len(only_in_lats)}): {only_in_lats}")
print(f"Genera only in se_est file ({len(only_in_se_est)}): {only_in_se_est}")

# Find out if the two columns are identical, in terms of their order specifically
model_8_rep_lats_file_test['genus'].equals(model_8_rep_se_est_file_test['species'])


Genera only in lats file (0): set()
Genera only in se_est file (0): set()


False

### Model 9

In [56]:
BRIDGE_syn_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\AF_climatic_data\\BRIDGE_data_new_dates\\synapsida_BRIDGE_z_trans_filtered_1myr_new_dates.txt"

model_9_syn_lats_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\Perm-Trias\\bdnn_trait_files\\pt_synapsida_lats_file_final.csv"
model_9_syn_se_est_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_9\\synapsida_new_dates\\combined_10_se_est.txt"
model_9_syn_per_species_rates_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_9\\synapsida_new_dates\\combined_10_per_species_rates.log"

model_9_syn = genera_data(model_9_syn_lats_file, 
            model_9_syn_se_est_file,
            model_9_syn_per_species_rates_file)

model_9_syn_nn = nearest_neighbor(BRIDGE_syn_file, model_9_syn)

model_9_syn_nn_wi = weighted_interpolation(BRIDGE_syn_file, model_9_syn_nn)

model_9_syn_nn_wi.head()

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lam,...,drymon_z_trans_NN,wetdry_z_trans_NN,mat_z_trans_WI,map_z_trans_WI,wmm_z_trans_WI,cmm_z_trans_WI,wmmcmm_z_trans_WI,wetmon_z_trans_WI,drymon_z_trans_WI,wetdry_z_trans_WI
0,Abajudon,-0.297425,0,0,1,0,266.160371,259.748497,6.411875,1.924390,...,1.221425,0.211046,-1.183661,0.847290,-1.357622,-0.532897,-0.147943,0.813748,1.221425,0.211046
1,Abdalodon,-0.297425,0,0,1,0,257.044265,256.831431,0.212834,0.786035,...,0.054401,-0.125815,-1.013536,-0.105045,1.082517,-1.479153,1.664987,-0.063729,0.054401,-0.125815
2,Acratophorus,-0.297425,0,0,1,0,233.318320,228.400394,4.917926,0.362145,...,0.948074,0.661316,0.171044,1.635367,0.051764,0.122606,-0.077466,0.999467,0.948074,0.661316
3,Adelobasileus,-0.297425,0,1,0,0,216.701009,216.199352,0.501657,0.840821,...,-0.783553,-0.799552,0.825377,-0.776213,1.145625,0.239217,0.295842,-1.013082,-0.783553,-0.799552
4,Aelurognathus,1.840695,0,0,1,0,260.162986,252.056465,8.106522,0.543822,...,0.055984,-0.109447,-1.020136,-0.171520,0.093622,-1.031190,0.877886,-0.050775,0.055984,-0.109447


In [57]:
model_9_syn_nn_wi.to_csv("C:/Users/SimoesLabAdmin/Documents/pt_diversity_rates/updated_occurrence_analyses/data/causality_data/model_9_syn_merged_nn_wi.csv", index=False)

In [ ]:
BRIDGE_rep_terr_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\AF_climatic_data\\BRIDGE_data_new_dates\\reptilia_terr_BRIDGE_z_trans_filtered_1myr_new_dates.txt"

model_9_rep_terr_lats_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\Perm-Trias\\bdnn_trait_files\\pt_reptilia_terr_lats_file_final.csv"
model_9_rep_terr_se_est_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_9\\reptilia_terr_new_dates\\combined_10_se_est.txt"
model_9_rep_terr_per_species_rates_file = "C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\model_9\\reptilia_terr_new_dates\\combined_10_per_species_rates.log"

model_9_rep_terr = genera_data(model_9_rep_terr_lats_file, 
            model_9_rep_terr_se_est_file,
            model_9_rep_terr_per_species_rates_file)

model_9_rep_terr_nn = nearest_neighbor(BRIDGE_rep_terr_file, model_9_rep_terr)

model_9_rep_terr_nn_wi = weighted_interpolation(BRIDGE_rep_terr_file, model_9_rep_terr_nn)

model_9_rep_terr_nn_wi.head()

,genus,lat_range_z_trans,Temperate_N,Tropical,Temperate_S,Antarctic,ts,te,time_span,avg_lam,...,drymon_z_trans_NN,wetdry_z_trans_NN,mat_z_trans_WI,map_z_trans_WI,wmm_z_trans_WI,cmm_z_trans_WI,wmmcmm_z_trans_WI,wetmon_z_trans_WI,drymon_z_trans_WI,wetdry_z_trans_WI
0,Abyssomedon,-0.298764,0,1,0,0,287.026971,286.840026,0.186945,0.423061,...,-1.810146,-0.849677,1.338319,-1.667559,0.618943,1.264486,-0.788962,-1.343780,-1.810146,-0.849677
1,Acadiella,-0.298764,0,1,0,0,233.151964,232.273919,0.878045,0.782552,...,-0.338950,0.925927,0.900076,0.506932,0.345128,0.919133,-0.632075,0.663140,-0.338950,0.925927
2,Acaenasuchus,-0.298764,0,1,0,0,226.671594,209.905381,16.766213,0.713145,...,-0.392124,0.864681,1.193378,0.479843,0.666223,1.147751,-0.657731,0.593073,-0.392124,0.864681
3,Acallosuchus,-0.298764,0,1,0,0,218.919006,217.995881,0.923125,0.265074,...,-0.392124,0.864681,1.193378,0.479843,0.666223,1.147751,-0.657731,0.593073,-0.392124,0.864681
4,Acerosodontosaurus,-0.298764,0,0,1,0,255.637319,255.535584,0.101735,1.150054,...,0.987963,-1.005904,-0.766474,-0.046076,1.691980,-1.769322,2.534502,-0.505604,0.987963,-1.005904


In [ ]:
model_9_rep_terr_nn_wi.to_csv("C:/Users/SimoesLabAdmin/Documents/pt_diversity_rates/updated_occurrence_analyses/data/causality_data/model_9_rep_terr_merged_nn_wi.csv", index=False)

### Success Check
Confirming that the functions worked just like the code in the causality_dataset_creation_EDA.ipynb

In [35]:
eda_nn = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\causality_data\\eda_outputs\\syn_merged_nn.csv")
eda_nn.rename(columns={"mean_pt_1myr_z_trans":"mean_pt_1myr_z_trans_NN_EDA", "Mod_R_deltaTMyr_pt_1myr_z_trans": "Mod_R_deltaTMyr_pt_1myr_z_trans_NN_EDA"}, inplace=True)

eda_wi = pd.read_csv("C:\\Users\\SimoesLabAdmin\\Documents\\pt_diversity_rates\\updated_occurrence_analyses\\data\\causality_data\\eda_outputs\\syn_merged_wi.csv")
eda_wi.rename(columns={"mean_pt_1myr_z_trans":"mean_pt_1myr_z_trans_WI_EDA", "Mod_R_deltaTMyr_pt_1myr_z_trans": "Mod_R_deltaTMyr_pt_1myr_z_trans_WI_EDA"}, inplace=True)

In [36]:
# Return true if they match
eda_nn_check = eda_nn["mean_pt_1myr_z_trans_NN_EDA"].equals(model_8_syn_nn["mean_pt_1myr_z_trans_NN"])
eda_wi_check = eda_wi["mean_pt_1myr_z_trans_WI_EDA"].equals(model_8_syn_nn_wi["mean_pt_1myr_z_trans_WI"])   
eda_nn_check, eda_wi_check

(False, False)

In [37]:
eda_nn_check = eda_nn["Mod_R_deltaTMyr_pt_1myr_z_trans_NN_EDA"].equals(model_8_syn_nn["Mod_R_deltaTMyr_pt_1myr_z_trans_NN"])
eda_wi_check = eda_wi["Mod_R_deltaTMyr_pt_1myr_z_trans_WI_EDA"].equals(model_8_syn_nn_wi["Mod_R_deltaTMyr_pt_1myr_z_trans_WI"])   
eda_nn_check, eda_wi_check

(False, False)

In [38]:
model_8_syn_check = model_8_syn_nn_wi[['genus', 'mean_pt_1myr_z_trans_NN', 'Mod_R_deltaTMyr_pt_1myr_z_trans_NN', 'mean_pt_1myr_z_trans_WI', 'Mod_R_deltaTMyr_pt_1myr_z_trans_WI']]
model_8_syn_check = pd.merge(model_8_syn_check, eda_nn[['genus', "mean_pt_1myr_z_trans_NN_EDA","Mod_R_deltaTMyr_pt_1myr_z_trans_NN_EDA"]], on='genus', how='left')
model_8_syn_check = pd.merge(model_8_syn_check, eda_wi[['genus', "mean_pt_1myr_z_trans_WI_EDA","Mod_R_deltaTMyr_pt_1myr_z_trans_WI_EDA"]], on='genus', how='left')
model_8_syn_check.head()

,genus,mean_pt_1myr_z_trans_NN,Mod_R_deltaTMyr_pt_1myr_z_trans_NN,mean_pt_1myr_z_trans_WI,Mod_R_deltaTMyr_pt_1myr_z_trans_WI,mean_pt_1myr_z_trans_NN_EDA,Mod_R_deltaTMyr_pt_1myr_z_trans_NN_EDA,mean_pt_1myr_z_trans_WI_EDA,Mod_R_deltaTMyr_pt_1myr_z_trans_WI_EDA
0,Abajudon,-0.917951,0.704704,-0.917951,0.704704,-0.917951,0.704704,-0.917951,0.704704
1,Abdalodon,-0.016673,-0.859216,-0.016673,-0.859216,-0.016673,-0.859216,-0.016673,-0.859216
2,Acratophorus,1.005908,0.025418,1.005908,0.025418,1.005908,0.025418,1.005908,0.025418
3,Adelobasileus,-0.162980,-0.763084,-0.074135,-0.093139,-0.162980,-0.763084,-0.074135,-0.093139
4,Aelurognathus,-0.544651,-0.226106,-0.544651,-0.226106,-0.544651,-0.226106,-0.544651,-0.226106


In [39]:
# reorder and rename b/c all these underscores are killing me

model_8_syn_check.rename(columns={'mean_pt_1myr_z_trans_NN': 'mean_NN', 'Mod_R_deltaTMyr_pt_1myr_z_trans_NN': 'Mod_NN', 'mean_pt_1myr_z_trans_WI': 'mean_WI', 'Mod_R_deltaTMyr_pt_1myr_z_trans_WI': 'Mod_WI', "mean_pt_1myr_z_trans_NN_EDA": "mean_NN_EDA", "Mod_R_deltaTMyr_pt_1myr_z_trans_NN_EDA": "Mod_NN_EDA", "mean_pt_1myr_z_trans_WI_EDA": "mean_WI_EDA", "Mod_R_deltaTMyr_pt_1myr_z_trans_WI_EDA": "Mod_WI_EDA"}, inplace=True)
model_8_syn_check.head()

,genus,mean_NN,Mod_NN,mean_WI,Mod_WI,mean_NN_EDA,Mod_NN_EDA,mean_WI_EDA,Mod_WI_EDA
0,Abajudon,-0.917951,0.704704,-0.917951,0.704704,-0.917951,0.704704,-0.917951,0.704704
1,Abdalodon,-0.016673,-0.859216,-0.016673,-0.859216,-0.016673,-0.859216,-0.016673,-0.859216
2,Acratophorus,1.005908,0.025418,1.005908,0.025418,1.005908,0.025418,1.005908,0.025418
3,Adelobasileus,-0.162980,-0.763084,-0.074135,-0.093139,-0.162980,-0.763084,-0.074135,-0.093139
4,Aelurognathus,-0.544651,-0.226106,-0.544651,-0.226106,-0.544651,-0.226106,-0.544651,-0.226106


In [28]:
model_8_syn_check[model_8_syn_check['mean_NN'] != model_8_syn_check['mean_NN_EDA']]

,genus,mean_NN,Mod_R_NN,mean_WI,Mod_R_WI,mean_NN_EDA,Mod_R_NN_EDA,mean_WI_EDA,Mod_R_WI_EDA
0,Abajudon,-0.917951,0.704704,-0.917951,0.704704,-0.917951,0.704704,-0.917951,0.704704
1,Abdalodon,-0.016673,-0.859216,-0.016673,-0.859216,-0.016673,-0.859216,-0.016673,-0.859216
2,Acratophorus,1.005908,0.025418,1.005908,0.025418,1.005908,0.025418,1.005908,0.025418
3,Adelobasileus,-0.162980,-0.763084,-0.074135,-0.093139,-0.162980,-0.763084,-0.074135,-0.093139
4,Aelurognathus,-0.544651,-0.226106,-0.544651,-0.226106,-0.544651,-0.226106,-0.544651,-0.226106
...,...,...,...,...,...,...,...,...,...
456,Woutersia,0.099289,-0.628074,0.099289,-0.628074,0.099289,-0.628074,0.099289,-0.628074
457,Woznikella,1.025593,-0.426136,1.025593,-0.426136,1.025593,-0.426136,1.025593,-0.426136
458,Xiyukannemeyeria,1.304997,0.275994,1.304997,0.275994,1.304997,0.275994,1.304997,0.275994
459,Yikezhaogia,0.814772,-0.796539,0.867735,-0.793066,0.814772,-0.796539,0.867735,-0.793066


In [29]:
(model_8_syn_check['mean_NN'] - model_8_syn_check['mean_NN_EDA']).describe()


count    4.610000e+02
mean     2.218533e-11
std      2.742682e-10
min     -4.972532e-10
25%     -2.434490e-10
50%      3.711143e-11
75%      2.578406e-10
max      4.848284e-10
dtype: float64

Ok so they only don't look equal because of floating point precision mismatch (I have a headache)

In [31]:
model_8_syn_check[model_8_syn_check['mean_NN'].round(6) != model_8_syn_check['mean_NN_EDA'].round(6)]

,genus,mean_NN,Mod_R_NN,mean_WI,Mod_R_WI,mean_NN_EDA,Mod_R_NN_EDA,mean_WI_EDA,Mod_R_WI_EDA


In [32]:
model_8_syn_check[model_8_syn_check['mean_WI'].round(6) != model_8_syn_check['mean_WI_EDA'].round(6)]

,genus,mean_NN,Mod_R_NN,mean_WI,Mod_R_WI,mean_NN_EDA,Mod_R_NN_EDA,mean_WI_EDA,Mod_R_WI_EDA


In [42]:
model_8_syn_check[model_8_syn_check['Mod_NN'].round(6) != model_8_syn_check['Mod_NN_EDA'].round(6)]

,genus,mean_NN,Mod_NN,mean_WI,Mod_WI,mean_NN_EDA,Mod_NN_EDA,mean_WI_EDA,Mod_WI_EDA


In [41]:
model_8_syn_check[model_8_syn_check['Mod_WI'].round(6) != model_8_syn_check['Mod_WI_EDA'].round(6)]

,genus,mean_NN,Mod_NN,mean_WI,Mod_WI,mean_NN_EDA,Mod_NN_EDA,mean_WI_EDA,Mod_WI_EDA


Success: the functions produced the exact same results as my by-hand eda notebook: causality_dataset_creation_EDA.ipynb